In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/workspaces/LLM_Zoomcamp_2026/llm-zoomcamp-code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1054.60it/s]


In [4]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [7]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [8]:
from tqdm.auto import tqdm

In [9]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

100%|██████████| 28/28 [01:57<00:00,  4.20s/it]


1400

In [10]:
import numpy as np
X = np.array(vectors)

X.shape

(1400, 384)

In [11]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [14]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [15]:
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client()

In [16]:
from rag_helper import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [17]:
vector_assistant.ask("the program has already begun, can I still sign up?")

'Yes, you can still join and start learning! However, if you want to receive a certificate, you need to make sure to submit your project while submissions are still being accepted. \n\nAre there other areas that you want to explore?'

In [19]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [20]:
vs_index.fit(vectors, documents)